In [ ]:
# imported libraries and path

In [ ]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings("ignore")
from math import sqrt
np.random.seed(42)
DATA_PATH = "emissions_by_country.csv"
TARGET_COL = "Total"

In [ ]:
#Reusable preprocessed pipeline

In [ ]:
class OutputPipeline:
    def __init__(self, scale_after_split: bool = False):
        self.settings = {}
        self.scale_after_split = scale_after_split
    def fit_transform(self, df, num_cols, cat_cols, target_col=None):
        clean_df = df.copy()
        target_series = None
        if target_col and target_col in clean_df.columns:
            target_series = clean_df[target_col].copy()
            clean_df = clean_df.drop(columns=[target_col])       
        for col in cat_cols:
            if col in clean_df.columns and clean_df[col].isnull().any():
                m = clean_df[col].mode(dropna=True)
                clean_df[col] = clean_df[col].fillna(m[0] if len(m) > 0 else "unknown")
        for col in num_cols:
            if col in clean_df.columns:
                mean_val = clean_df[col].mean()
                clean_df[col] = clean_df[col].fillna(mean_val)
        for col in cat_cols:
            if col in clean_df.columns:
                clean_df[col] = clean_df[col].astype(str).str.strip().str.lower()       
        if len(num_cols) > 0:
            zscores = np.abs((clean_df[num_cols] - clean_df[num_cols].mean()) / clean_df[num_cols].std())
            for c in num_cols:
                if c in clean_df.columns:
                    median_val = clean_df[c].median()
                    outliers = zscores[c] > 3
                    clean_df.loc[outliers, c] = median_val
        binary_cols, onehot_cols = [], []
        for col in list(cat_cols):
            if col not in clean_df.columns:
                continue
            unique_vals = clean_df[col].nunique(dropna=True)
            if unique_vals == 2:
                mapping = {val: i for i, val in enumerate(clean_df[col].unique())}
                clean_df[col] = clean_df[col].map(mapping)
                binary_cols.append(f"{col} (binary: {mapping})")
            elif 2 < unique_vals <= 50:
                dummies = pd.get_dummies(clean_df[col], prefix=col, drop_first=True)
                clean_df = pd.concat([clean_df.reset_index(drop=True), dummies.reset_index(drop=True)], axis=1)
                clean_df.drop(columns=[col], inplace=True)
                onehot_cols.append(col)       
        if not self.scale_after_split:
            for c in num_cols:
                if c in clean_df.columns:
                    mean = clean_df[c].mean()
                    std = clean_df[c].std()
                    clean_df[c] = (clean_df[c] - mean) / (std if std > 0 else 1)
        for col in clean_df.columns:
            if clean_df[col].dtype == "object":
                try:
                    clean_df[col] = pd.to_numeric(clean_df[col])
                except Exception:
                    pass
        for col in clean_df.columns:
            if clean_df[col].isnull().any():
                if np.issubdtype(clean_df[col].dtype, np.number):
                    mean_val = clean_df[col].mean()
                    clean_df[col] = clean_df[col].fillna(mean_val if not np.isnan(mean_val) else 0)
                else:
                    clean_df[col] = clean_df[col].fillna("unknown")
        if target_series is not None:
            clean_df[target_col] = target_series
        self.settings = {
            "missing_values": "Mode (categorical), Mean (numeric)",
            "outlier_strategy": "Z-score > 3 → median",
            "binary_encoded": binary_cols,
            "onehot_encoded": onehot_cols,
            "scaled_columns": [] if self.scale_after_split else num_cols,
            "scaling": "skipped here; done after split" if self.scale_after_split else "z-score (here)",
            "final_missing_filled": True,
            "target_preserved": bool(target_col)
        }
        return clean_df
    def summary(self):
        return self.settings

In [ ]:
#description of data before preprocessing

In [ ]:

df = pd.read_csv(DATA_PATH)
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
print("Missing target rows BEFORE drop:", df[TARGET_COL].isna().sum())
df = df.dropna(subset=[TARGET_COL]).reset_index(drop=True)
print("Rows after dropping missing target:", len(df))
num_cols_df = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
cat_cols_df = [c for c in df.columns if c not in num_cols_df]
print(" Dataset Description ==")
print(" Domain: Country/year emissions")
print("Samples: ",{len(df)})
print("Target: {TARGET_COL}")
print(df.head(3))


# data prearation

In [ ]:
def split_idx(n, test_ratio=0.2, seed=42):
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)
    t = int(np.floor(test_ratio * n))
    return idx[t:], idx[:t]
train_idx, test_idx = split_idx(len(df), 0.2, 42)
df_tr = df.iloc[train_idx].reset_index(drop=True)
df_te = df.iloc[test_idx].reset_index(drop=True)
#This helps to keep track of which rows belong to train/test after you later concatenate them for preprocessing
marker_tr = df_tr.copy(); 
marker_tr["_split_"] = "train"
marker_te = df_te.copy(); 
marker_te["_split_"] = "test"
combo = pd.concat([marker_tr, marker_te], axis=0, ignore_index=True)
num_X_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]) and c != TARGET_COL]
cat_X_cols = [c for c in df.columns if c not in num_X_cols + [TARGET_COL, "_split_"]]
pipe = OutputPipeline(scale_after_split=True) 
combo_clean = pipe.fit_transform(combo, num_cols=num_X_cols, cat_cols=cat_X_cols, target_col=TARGET_COL)
print("Pipeline summary:", pipe.summary())
df_tr_clean = combo_clean[combo_clean["_split_"]=="train"].drop(columns=["_split_"]).reset_index(drop=True)
df_te_clean = combo_clean[combo_clean["_split_"]=="test"].drop(columns=["_split_"]).reset_index(drop=True)
feature_cols = [c for c in df_tr_clean.columns if c != TARGET_COL]
X_train = df_tr_clean[feature_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
X_test  = df_te_clean[feature_cols].apply(pd.to_numeric, errors="coerce").to_numpy(dtype=float)
y_train = pd.to_numeric(df_tr_clean[TARGET_COL], errors="coerce").to_numpy(dtype=float).reshape(-1,1)
y_test  = pd.to_numeric(df_te_clean[TARGET_COL], errors="coerce").to_numpy(dtype=float).reshape(-1,1)
print("Shapes BEFORE scaling:",
      "X_train", X_train.shape, "y_train", y_train.shape,
      "X_test", X_test.shape, "y_test", y_test.shape)


# Scaling is done after split to ensure no leakage 

In [ ]:

def zscore_fit(X):
    m = X.mean(axis=0)
    s = X.std(axis=0)
    s[s==0] = 1.0
    return m, s
def zscore_apply(X, m, s):
    return (X - m) / s
mu_tr, sig_tr = zscore_fit(X_train)
X_train_z = zscore_apply(X_train, mu_tr, sig_tr)
X_test_z  = zscore_apply(X_test,  mu_tr, sig_tr)
# This we used to ensure if any leftover with 0 (mean)
X_train_z = np.nan_to_num(X_train_z, nan=0.0, posinf=0.0, neginf=0.0)
X_test_z  = np.nan_to_num(X_test_z,  nan=0.0, posinf=0.0, neginf=0.0)
# This we used to add bias to the code
def add_bias(X): 
    return np.hstack([X, np.ones((X.shape[0],1), dtype=float)])
X_train_b = add_bias(X_train_z)
X_test_b  = add_bias(X_test_z)
print("Shapes after scaling:",
      "X_train_b", X_train_b.shape, "y_train", y_train.shape,
      "X_test_b", X_test_b.shape, "y_test", y_test.shape)


# initialization of xavier 

In [ ]:
def xavier_initialization(input_dim: int, output_dim: int = 1, seed: int = 42):
    rng = np.random.default_rng(seed)
    limit = np.sqrt(6.0 / (input_dim + output_dim))
    return rng.uniform(-limit, limit, size=(input_dim, output_dim))

# Implemented Elastic Net Regularization
# Solved the problem using Gradient Descent. 
# Implement gradient descent from scratch to
# minimized the Elastic Net regularized loss function.
# Updated the weights 
# Experimented with at least three different methods for weights initializing. Some ideas:
# a. Random: initialize weights randomly
# b.zero: initialize all weights to zero
# c.Xavier (Glorot) Initialization: initialize weights in such a way that helps to



In [ ]:
class ElasticGD:
    def __init__(self, lr=1e-3, max_iters=20000, lam_l2=0.01, lam_l1=0.001,init="xavier", stop_rule="iterations", grad_tol=1e-4, seed=42):
        self.lr = lr
        self.max_iters = max_iters
        self.lam_l2 = lam_l2
        self.lam_l1 = lam_l1
        self.init = init              
        self.stop_rule = stop_rule     
        self.grad_tol = grad_tol
        self.seed = seed
        self.w = None
        self.loss_history = []
    def _init_w(self, d):
        if self.init == "zeros":
            return np.zeros((d, 1))
        if self.init == "random":
            rng = np.random.default_rng(self.seed)
            return rng.normal(0, 0.01, size=(d, 1))
        return xavier_initialization(d, 1, seed=self.seed)
    def _loss(self, Xb, y, w):
        N = Xb.shape[0]
        resid = (Xb @ w) - y
        data = (resid**2).sum() / (2.0 * N)
        l2 = (self.lam_l2 / 2.0) * (w[:-1]**2).sum()     
        l1 = self.lam_l1 * np.abs(w[:-1]).sum()          
        return float(data + l2 + l1)
    def _grad(self, Xb, y, w):
        N = Xb.shape[0]
        resid = (Xb @ w) - y
        g = (Xb.T @ resid) / N
        g[:-1] += self.lam_l2 * w[:-1]                  
        g[:-1] += self.lam_l1 * np.sign(w[:-1])          
        return g
    def fit(self, Xb, y):
        d = Xb.shape[1]
        self.w = self._init_w(d)
        self.loss_history = []
        for _ in range(self.max_iters):
            g = self._grad(Xb, y, self.w)                
            self.w -= self.lr * g                         
            self.loss_history.append(self._loss(Xb, y, self.w))
            if self.stop_rule == "grad_norm" and np.linalg.norm(g) < self.grad_tol:
                break
        return self
    def predict(self, Xb):
        return Xb @ self.w
def mse(y, yhat): 
    return float(np.mean((y - yhat)**2))
def r2(y, yhat):
    ss_res = float(np.sum((y - yhat)**2))
    ss_tot = float(np.sum((y - np.mean(y))**2))
    return 1.0 - ss_res/ss_tot if ss_tot > 0 else 0.0

# 7. Experiment with training stopping criteria (≥2 methods):

In [ ]:
mdl1 = ElasticGD(lr=1e-3, max_iters=30000, lam_l2=0.01, lam_l1=0.001,init="zeros", stop_rule="iterations", grad_tol=1e-4, seed=42).fit(X_train_b, y_train)
ytr1 = mdl1.predict(X_train_b)
yte1 = mdl1.predict(X_test_b)
metrics1 = {
    "setup": "zero+fixedIters",
    "train_MSE": mse(y_train, ytr1),
    "test_MSE":  mse(y_test,  yte1),
    "train_R2":  r2(y_train, ytr1),
    "test_R2":   r2(y_test,  yte1),
    "iters":     len(mdl1.loss_history)
}
print(metrics1)

In [ ]:
mdl2= ElasticGD(lr=1e-3, max_iters=30000, lam_l2=0.01, lam_l1=0.001,init="random", stop_rule="grad_norm", grad_tol=1e-4, seed=42).fit(X_train_b, y_train)
ytr2= mdl2.predict(X_train_b)
yte1 = mdl2.predict(X_test_b)
metrics2 = {
    "setup": "random+gradNorm",
    "train_MSE": mse(y_train, ytr2),
    "test_MSE":  mse(y_test,  yte1),
    "train_R2":  r2(y_train, ytr2),
    "test_R2":   r2(y_test,  yte1),
    "iters":     len(mdl2.loss_history)
}
print(metrics2)

In [ ]:
mdl3 = ElasticGD(lr=1e-3, max_iters=30000, lam_l2=0.01, lam_l1=0.001,init="xavier", stop_rule="iterations", grad_tol=1e-4, seed=42).fit(X_train_b, y_train)
ytr3 = mdl3.predict(X_train_b)
yte3 = mdl3.predict(X_test_b)
metrics3 = {
    "setup": "xavier+fixedIters",
    "train_MSE": mse(y_train, ytr3),
    "test_MSE":  mse(y_test,  yte3),
    "train_R2":  r2(y_train, ytr3),
    "test_R2":   r2(y_test,  yte3),
    "iters":     len(mdl3.loss_history)
}
print(metrics3)

In [ ]:
runs_7 = [
    ("zero:",   mdl1, metrics1),
    ("radNorm",   mdl2, metrics2),
    ("xavier", mdl3, metrics3),
]
summary_7 = pd.DataFrame([m for _, _, m in runs_7]).sort_values("test_MSE")
print(" Comparison:")
print(summary_7.to_string(index=False))
best_name, best_model, best_metrics = min(runs_7, key=lambda t: t[2]["test_MSE"])
fastest_name = min(runs_7, key=lambda t: t[2]["iters"])[0]

print("Quick takeaways to paste into analysis:")
print(" Best generalisation :",best_name)
print(" Fastest convergence :", {fastest_name})

In [ ]:
best_name, best_model, best_metrics = min(runs_7, key=lambda t: t[2]["test_MSE"])
WEIGHTS_PATH = "a1_part4_weights_odrabira_pranayla.pickle"
with open(WEIGHTS_PATH, "wb") as f:
    pickle.dump({
        "weights": best_model.w,
        "feature_order": feature_cols + ["<BIAS>"],
        "train_mean": mu_tr,
        "train_std": sig_tr,
        "config": best_name,
        "metrics": best_metrics
    }, f)
print("Saved best weights to:", WEIGHTS_PATH)
print("Best setup:", best_name)
print("Best metrics:", best_metrics)

# Evaluation and plots 

In [ ]:
def get_best_model():    
    if 'runs' in globals() and runs and 'test_mse' in runs[0]:
        i = int(np.argmin([r["test_mse"] for r in runs]))
        r = runs[i]
        return r["w"], r["loss_curve"], r.get("config","best_from_runs"), r["train_mse"], r["train_r2"], r["test_mse"], r["test_r2"]
    if 'runs_7' in globals() and runs_7:
        name, model, metrics = min(runs_7, key=lambda t: t[2]["test_MSE"])
        return model.w, model.loss_history, name, metrics["train_MSE"], metrics["train_R2"], metrics["test_MSE"], metrics["test_R2"]
    model = ElasticGD(lr=1e-3, max_iters=30000, lam_l2=0.01, lam_l1=0.001,
                         init="xavier", stop_rule="iterations", grad_tol=1e-4, seed=42).fit(X_train_b, y_train)
    ytr = model.predict(X_train_b); yte = model.predict(X_test_b)
    return model.w, model.loss_history, "default_xavier", mse(y_train,ytr), r2(y_train,ytr), mse(y_test,yte), r2(y_test,yte)
w_best, loss_best, best_name, tr_mse, tr_r2, te_mse, te_r2 = get_best_model()
print("Evaluation")
print("Best setup:", best_name)
print(f"Train: MSE={tr_mse:.6f}, R²={tr_r2:.4f}")
print(f"Test : MSE={te_mse:.6f}, R²={te_r2:.4f}")

In [ ]:
yhat_test = X_test_b @ w_best
plt.figure()
plt.scatter(y_test.flatten(), yhat_test.flatten(), s=14)
lo, hi = float(np.min(y_test)), float(np.max(y_test))
plt.plot([lo, hi], [lo, hi])  # y = x line
plt.xlabel("Actual")
plt.ylabel("Predicted")
plt.title(" Predictions vs Actuals (Test)")
plt.show()

In [ ]:
plt.figure()
plt.plot(loss_best)
plt.xlabel("Iteration")
plt.ylabel("Loss")
plt.title("Loss vs Iterations (Best Run)")
plt.show()

In [ ]:
def fit_and_mse(l2, l1):
    model = ElasticGD(lr=5e-4,max_iters=12_000,lam_l2=l2, lam_l1=l1,init="xavier",stop_rule="grad_norm",grad_tol=1e-3,seed=42).fit(X_train_b, y_train)
    return mse(y_test, model.predict(X_test_b))
lam_l2_grid = [0.0, 1e-3, 1e-2, 1e-1]
lam_l1_grid = [0.0, 1e-4, 1e-3]

mse_matrix = np.zeros((len(lam_l2_grid), len(lam_l1_grid)))
for i, l2 in enumerate(lam_l2_grid):
    for j, l1 in enumerate(lam_l1_grid):
        mse_matrix[i, j] = fit_and_mse(l2, l1)

In [ ]:
plt.figure()
plt.imshow(np.log10(np.clip(mse_matrix, 1e-12, np.inf)), aspect="auto")
plt.colorbar(label="log10(Test MSE)")
plt.xticks(range(len(lam_l1_grid)), [f"{v:g}" for v in lam_l1_grid], rotation=30)
plt.yticks(range(len(lam_l2_grid)), [f"{v:g}" for v in lam_l2_grid])
plt.xlabel("λ1 (L1 strength)")
plt.ylabel("λ2 (L2 strength)")
plt.title("log10(Test MSE) vs λ grid")
plt.show()

In [ ]:
plt.figure()
for i, l2 in enumerate(lam_l2_grid):
    plt.plot(lam_l1_grid, mse_matrix[i, :], marker="o", label=f"λ2={l2:g}")
plt.yscale("log")
plt.xlabel("λ1 (L1 strength)")
plt.ylabel("Test MSE (log)")
plt.title(" FAST: Test MSE vs λ1 for different λ2")
plt.legend(); plt.tight_layout(); plt.show()

## Analysis and Report:
# Elastic Net's performance in comparison to Ridge and Lasso varied between my two datasets. Elastic Net with Xavier initialization and GD only achieved about 0.32–0.35, whereas Ridge produced the strongest results for one dataset with R2 around 0.71, demonstrating that Ridge handled the data structure better. Elastic Net outperformed the other dataset because it was able to eliminate irrelevant features and reduce correlated ones. This demonstrates that Elastic Net does not always ensure an improvement because its efficacy is highly dependent on the noise and feature correlations in the dataset.

## Benefits:
# Elastic Net with Gradient Descent's strength is its balance; it can scale to large datasets without the need for matrix inversion, combining the stability of Ridge with the feature selection of Lasso. Because of this, it is particularly helpful in high-dimensional environments. I did observe some drawbacks, though: convergence was slower than with closed-form Ridge, performance was noticeably worse in one instance, and it required careful hyperparameter tuning (learning rate, λ, α). Depending on the dataset, Elastic Net + GD may not always perform better than more straightforward models, but it is generally strong when feature selection and correlation handling are required.

In [ ]:
try:
    feature_order = feature_cols + ["<BIAS>"]
except:
    feature_order = [f"f{i}" for i in range(X_train_b.shape[1]-1)] + ["<BIAS>"]
try:
    train_mean, train_std = mu_tr, sig_tr
except:
    train_mean = np.zeros(X_train_b.shape[1]-1)
    train_std  = np.ones(X_train_b.shape[1]-1)
SAVE_PATH = "a1_part4_weights_odrabira_pranayla.pickle"
with open(SAVE_PATH, "wb") as f:
    pickle.dump({
        "weights": w_best,
        "feature_order": feature_order,
        "train_mean": train_mean,
        "train_std": train_std,
        "config": best_name,
        "metrics": {"train_MSE": tr_mse, "train_R2": tr_r2, "test_MSE": te_mse, "test_R2": te_r2}
    }, f)
print("Saved best model to:", SAVE_PATH)

| Team Member | Assignment part | Contribution |
|----------|----------|----------|
| Odra Bira | Part 3  | 50% |
| Pranay Lachuluri | Part 3 | 50% |
| Odra Bira | Part 4  | 50% |
| Pranay Lachuluri | Part 4| 50% |


#REFERENCES:
1.⁠https://www.geeksforgeeks.org/deep-learning/xavier-initialization/
2.https://www.geeksforgeeks.org/machine-learning/implementation-of-elastic-net-regression-from-scratch/
3.https://matplotlib.org/stable/contents.html